# Predictive Analytics — Construction Worker Accident Prediction

<h2 style='color:green' align='center'>Objective: To predict whether a construction worker will experience an accident using Logistic Regression</h2>

**Tasks:**
1. Train a logistic regression model to predict whether a construction worker will experience an accident
2. Evaluate classification performance (Confusion Matrix, Accuracy, Precision, Recall, F1-Score, ROC-AUC)
3. Predict the probability of an accident for workers with different characteristics
4. Assess which variables significantly influence accident probability

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

---
## 2. Load and Explore the Data

In [ ]:
# Load dataset
df = pd.read_csv('construction_workers_accidents.csv')
df.head(10)

In [ ]:
# Dataset shape
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Variable types and non-null counts
df.info()

In [ ]:
# Descriptive statistics
df.describe().round(2)

In [ ]:
# Target variable distribution
print("Target class distribution (accident_status):")
print(df['accident_status'].value_counts())
print()
print(df['accident_status'].value_counts(normalize=True).round(3) * 100, "(%)")

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

features = ['gender', 'age', 'education', 'harness', 'training', 'BMI', 'heartRate']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4', '#8BC34A']

for i, (feat, color) in enumerate(zip(features, colors)):
    df.groupby('accident_status')[feat].plot(
        kind='hist', alpha=0.6, ax=axes[i], bins=20, legend=True, color=[color, '#E0E0E0']
    )
    axes[i].set_title(f'{feat} by Accident Status', fontsize=11)
    axes[i].set_xlabel(feat)
    axes[i].legend(['No Accident (0)', 'Accident (1)'], fontsize=8)

# Correlation heatmap in last subplot
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[7], linewidths=0.5)
axes[7].set_title('Correlation Matrix', fontsize=11)

plt.suptitle('EDA: Feature Distributions & Correlation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Handle Missing Values

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isna().sum())
print()

# Impute with median (robust to outliers for continuous), mode for categorical/ordinal
# education (ordinal) → mode; BMI, heartRate (continuous) → median
df = df.assign(
    education=df['education'].fillna(df['education'].mode()[0]),
    BMI=df['BMI'].fillna(df['BMI'].median()),
    heartRate=df['heartRate'].fillna(df['heartRate'].median())
)

print("After imputation — missing values:")
print(df.isna().sum())

---
## 5. Feature and Target Definition

In [ ]:
# Define features (X) and target (y)
feature_cols = ['gender', 'age', 'education', 'harness', 'training', 'BMI', 'heartRate']
X = df[feature_cols]
y = df['accident_status']

print(f"Features shape: {X.shape}")
print(f"Target shape  : {y.shape}")
print(f"Feature columns: {feature_cols}")

---
## 5. Train-Test Split

In [ ]:
# Split: 80% train, 20% test (random_state for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")
print(f"Train class ratio: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test class ratio : {y_test.value_counts(normalize=True).round(3).to_dict()}")

---
## 6. Feature Scaling

> Logistic Regression benefits from scaled features, especially when continuous variables (age, BMI, heartRate) are on different scales than binary/ordinal variables.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # Only transform test (no leakage)

print("Feature scaling applied (StandardScaler).")
print(f"Scaled X_train shape: {X_train_scaled.shape}")

---
## Task 1 — Train the Logistic Regression Model

In [ ]:
# Instantiate and train the model
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")
print(f"Model intercept : {log_model.intercept_[0]:.4f}")
print()
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': log_model.coef_[0]})
print(coef_df.to_string(index=False))

In [ ]:
# Predict on test set
y_pred      = log_model.predict(X_test_scaled)
y_pred_prob = log_model.predict_proba(X_test_scaled)[:, 1]   # Probability of class 1

print("Predictions (first 10):", y_pred[:10])
print("Probabilities (first 10):", y_pred_prob[:10].round(3))

---
## Task 2 — Evaluate Classification Performance

### 2i. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Accident (0)', 'Accident (1)'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (TN) : {tn}  — Correctly predicted NO accident")
print(f"False Positives (FP): {fp}  — Predicted accident, actually NO accident")
print(f"False Negatives (FN): {fn}  — Predicted NO accident, actually HAD accident")
print(f"True Positives (TP) : {tp}  — Correctly predicted accident")

### 2ii. Accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}  ({accuracy*100:.2f}%)")
print("\nInterpretation: The model correctly classifies",
      f"{accuracy*100:.2f}% of all workers in the test set.")

### 2iii. Precision

In [ ]:
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.4f}  ({precision*100:.2f}%)")
print("\nInterpretation: Of all workers predicted to have an accident,",
      f"{precision*100:.2f}% actually did.")

### 2iv. Recall (Sensitivity)

In [ ]:
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.4f}  ({recall*100:.2f}%)")
print("\nInterpretation: Of all workers who actually had an accident,",
      f"the model correctly identified {recall*100:.2f}%.")

### 2v. F1-Score

In [ ]:
f1 = f1_score(y_test, y_pred)
print(f"F1-Score: {f1:.4f}")
print("\nInterpretation: F1 is the harmonic mean of Precision and Recall.",
      "A higher F1 reflects better balance between the two.")

### 2vi. ROC-AUC

In [ ]:
roc_auc = roc_auc_score(y_test, y_pred_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='orange')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curve — Logistic Regression', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ROC-AUC Score: {roc_auc:.4f}")
print("\nInterpretation: AUC closer to 1.0 indicates a better discriminating model.")

### Summary of All Metrics

In [ ]:
metrics = {
    'Metric'   : ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Score'    : [accuracy, precision, recall, f1, roc_auc],
    'Score (%)':[f"{v*100:.2f}%" for v in [accuracy, precision, recall, f1, roc_auc]]
}
metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(metrics['Metric'], metrics['Score'],
               color=['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336'])
ax.set_xlim(0, 1.1)
ax.set_xlabel('Score', fontsize=12)
ax.set_title('Model Performance Summary', fontsize=13, fontweight='bold')
for bar, score in zip(bars, metrics['Score']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=10)
ax.axvline(x=0.5, color='gray', linestyle='--', lw=1, alpha=0.7)
plt.tight_layout()
plt.show()

---
## Task 3 — Predict Accident Probability for Workers with Different Characteristics

In [ ]:
# Define workers with contrasting profiles
# Features: gender, age, education, harness, training, BMI, heartRate
new_workers = pd.DataFrame({
    'gender'   : [1,    0,    1,    0,    1   ],
    'age'      : [25,   55,   35,   48,   62  ],
    'education': [4,    1,    3,    2,    1   ],
    'harness'  : [1,    0,    1,    0,    0   ],
    'training' : [3,    1,    2,    3,    1   ],
    'BMI'      : [22.0, 31.5, 25.0, 28.0, 33.0],
    'heartRate': [72,   95,   80,   88,   100 ]
})

# Scale and predict
new_workers_scaled = scaler.transform(new_workers)
pred_classes       = log_model.predict(new_workers_scaled)
pred_probs         = log_model.predict_proba(new_workers_scaled)[:, 1]

# Results table
results = new_workers.copy()
results['Predicted Probability (Accident)'] = pred_probs.round(4)
results['Predicted Class'] = pred_classes
results['Outcome'] = results['Predicted Class'].map({0: 'No Accident', 1: 'Accident'})

print(results.to_string(index=False))

In [ ]:
# Visual: predicted probability per worker profile
labels = [f"Worker {i+1}" for i in range(len(results))]
colors_bar = ['#F44336' if p >= 0.5 else '#4CAF50' for p in pred_probs]

plt.figure(figsize=(8, 4))
bars = plt.bar(labels, pred_probs, color=colors_bar, edgecolor='black', linewidth=0.7)
plt.axhline(y=0.5, color='navy', linestyle='--', label='Decision Threshold (0.5)')
for bar, prob in zip(bars, pred_probs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{prob:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel('Predicted Probability of Accident', fontsize=11)
plt.title('Accident Probability for Different Worker Profiles', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("\nRed bars = Predicted Accident | Green bars = Predicted No Accident")

---
## Task 4 — Assess Which Variables Significantly Influence Accident Probability

In [ ]:
# Logistic regression coefficients (standardised scale = comparable importance)
coef_df = pd.DataFrame({
    'Feature'    : feature_cols,
    'Coefficient': log_model.coef_[0],
    'Odds Ratio' : np.exp(log_model.coef_[0])
}).sort_values('Coefficient', key=abs, ascending=False)

coef_df['Direction'] = coef_df['Coefficient'].apply(lambda x: 'Increases Risk ↑' if x > 0 else 'Decreases Risk ↓')
print(coef_df.round(4).to_string(index=False))

In [ ]:
# Feature Importance Plot (by standardised coefficient magnitude)
coef_sorted = coef_df.sort_values('Coefficient')
bar_colors  = ['#F44336' if c > 0 else '#2196F3' for c in coef_sorted['Coefficient']]

plt.figure(figsize=(9, 5))
bars = plt.barh(coef_sorted['Feature'], coef_sorted['Coefficient'],
                color=bar_colors, edgecolor='black', linewidth=0.6)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Standardised Coefficient (Log-Odds)', fontsize=12)
plt.title('Feature Influence on Accident Probability\n(Red = Increases Risk | Blue = Decreases Risk)',
          fontsize=12, fontweight='bold')
for bar, coef in zip(bars, coef_sorted['Coefficient']):
    xpos = bar.get_width() + 0.002 if coef >= 0 else bar.get_width() - 0.002
    ha   = 'left' if coef >= 0 else 'right'
    plt.text(xpos, bar.get_y() + bar.get_height()/2,
             f'{coef:.3f}', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Odds Ratio plot — easier to interpret in safety context
or_sorted = coef_df.sort_values('Odds Ratio', ascending=True)
or_colors = ['#F44336' if o > 1 else '#2196F3' for o in or_sorted['Odds Ratio']]

plt.figure(figsize=(9, 5))
bars = plt.barh(or_sorted['Feature'], or_sorted['Odds Ratio'],
                color=or_colors, edgecolor='black', linewidth=0.6)
plt.axvline(x=1, color='black', linestyle='--', linewidth=1.2, label='OR = 1 (no effect)')
plt.xlabel('Odds Ratio', fontsize=12)
plt.title('Odds Ratio per Feature\n(OR > 1 → Increases Accident Odds | OR < 1 → Reduces Accident Odds)',
          fontsize=12, fontweight='bold')
for bar, od in zip(bars, or_sorted['Odds Ratio']):
    plt.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{od:.3f}', va='center', fontsize=9)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("\nKey Findings:")
for _, row in coef_df.iterrows():
    print(f"  {row['Feature']:10s} | Coef: {row['Coefficient']:+.4f} | OR: {row['Odds Ratio']:.4f} | {row['Direction']}")

---
## 7. Full Results Table

In [ ]:
# Save full prediction results on test set
X_test_df = pd.DataFrame(X_test.values, columns=feature_cols)
results_table = X_test_df.copy()
results_table['y_actual']              = y_test.values
results_table['predicted_probability'] = y_pred_prob.round(4)
results_table['y_predicted']           = y_pred
results_table['correct']               = (results_table['y_actual'] == results_table['y_predicted'])

results_table.head(15)

In [ ]:
# Save to CSV
results_table.to_csv('construction_accident_predictions.csv', index=False)
print("Results saved to 'construction_accident_predictions.csv'")

---
## Summary

| Task | Finding |
|------|----------|
| **Model** | Multivariate Logistic Regression on 7 features |
| **Accuracy** | Calculated above |
| **Precision** | Proportion of predicted accidents that were real |
| **Recall** | Proportion of actual accidents correctly detected |
| **F1-Score** | Harmonic mean of Precision & Recall |
| **ROC-AUC** | Overall discriminative ability of the model |
| **Key Risk Factors** | See Feature Importance plot above — variables with largest positive coefficients most strongly increase accident risk |

**Note on Odds Ratios:**  
- OR > 1 → the feature *increases* the odds of an accident  
- OR < 1 → the feature *decreases* the odds of an accident  
- OR = 1 → no effect

---
# The End